# DESIGN THE KEY — de novo mutant-specific binder (RUNG-26, v2: REAL executable code)

**GPU REQUIRED. ~4 h. v1 shipped scaffolds (instructions) for the design step → 0 designs. v2 runs ACTUAL binder design via ColabDesign/AfDesign (scriptable AF2-hallucination).**

Target: clean neoantigen **IDH1-R132H / HLA-A\*01:01**. Design a mini-binder that grips the **mutant** pMHC (hotspot forced on the mutated His@peptide-pos-4) and **not** the WT. Each binder: design vs MUT (AfDesign) → predict same binder vs WT → discrimination (i_pae_WT − i_pae_MUT).

**Rule-5 gate:** Cell 4 runs ONE real short design (~5–10 min). It must output a `{plddt, i_pae}` dict. Only then run the 4-h batch (Cell 5) — time-boxed + resumable. **Set Runtime → GPU.**

In [ ]:
#@title Cell 1 — clone + GPU + spec
import os, json, subprocess
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L || print('!! NO GPU — Runtime>GPU')
SPEC=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec','IDH1_R132H_A0101'],capture_output=True,text=True).stdout)
globals().update(GROOVE=SPEC['mhc_groove'], PEP_MUT=SPEC['pep_mut'], PEP_WT=SPEC['pep_wt'], MUTPOS=SPEC['mut_residue_in_peptide'])
print('target:',SPEC['gene'],SPEC['mut_label'],SPEC['allele'],'| mut',PEP_MUT,'WT',PEP_WT,'| hotspot pep pos',MUTPOS)
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — validate orchestration (selftest, no GPU)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung26_binder_design', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/52_binder_design.py','selftest'], RUNLOG)
print('[CELL 2]','✓' if rc==0 else f'✗ {rc} stop')

In [ ]:
#@title Cell 3 — fold MUT/WT pMHC targets with BOLTZ (proven in RUNG-20) → PDB  [~20 min, GPU]
import os, glob, subprocess, importlib.util as iu
try:
    from google.colab import drive; drive.mount('/content/drive'); WORK='/content/drive/MyDrive/cancer-recon/rung26'
except Exception: WORK='/content/rung26'
os.makedirs(WORK, exist_ok=True); os.environ['RUNG26_WORK']=WORK
if iu.find_spec('boltz') is None: !pip -q install boltz
if iu.find_spec('gemmi') is None: !pip -q install gemmi
import gemmi
def fold_pmhc(kind, pep):
    pdb=f'{WORK}/pmhc_{kind}.pdb'
    if os.path.exists(pdb): print(f'  {kind}: cached', pdb); return pdb
    y=f'{WORK}/pmhc_{kind}.yaml'; outd=f'{WORK}/boltz_{kind}'
    open(y,'w').write('sequences:\n  - protein:\n      id: A\n      sequence: '+GROOVE+'\n  - protein:\n      id: B\n      sequence: '+pep+'\n')
    !boltz predict {y} --use_msa_server --out_dir {outd} 2>&1 | tail -3
    cifs=sorted(glob.glob(f'{outd}/**/*_model_0.cif', recursive=True)) or sorted(glob.glob(f'{outd}/**/*.cif', recursive=True))
    if not cifs: print(f'  {kind}: NO CIF — check boltz output'); return None
    st=gemmi.read_structure(cifs[0]); st.setup_entities(); st.write_pdb(pdb)
    print(f'  {kind}: {cifs[0]} -> {pdb}'); return pdb
MUT_PDB=fold_pmhc('mut', PEP_MUT); WT_PDB=fold_pmhc('wt', PEP_WT)
assert MUT_PDB and WT_PDB and os.path.exists(MUT_PDB) and os.path.exists(WT_PDB), 'target fold failed — fix before Cell 4'
globals().update(MUT_PDB=MUT_PDB, WT_PDB=WT_PDB)
print('[CELL 3] ✓ pMHC targets (Boltz):', MUT_PDB, '|', WT_PDB)

In [ ]:
#@title Cell 4 — install ColabDesign (AfDesign) + SMOKE one REAL short design  [~10 min, GPU]
import os, glob, importlib.util as iu
if iu.find_spec('colabdesign') is None: !pip -q install git+https://github.com/sokrypton/ColabDesign.git
if not glob.glob('params/params_model_*.npz'):
    !mkdir -p params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar 2>/dev/null || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar)
    !tar -xf alphafold_params_2022-12-06.tar -C params && rm -f alphafold_params_2022-12-06.tar
from colabdesign import mk_afdesign_model, clear_mem
HOTSPOT=f'B{MUTPOS}'   # mutated peptide residue (chain B) — forces the binder to read the mutation
def _metrics(m):
    log=m.aux['log']; return {'i_pae':float(log.get('i_pae',log.get('pae',99))), 'plddt':float(log.get('plddt',0))*100}
def design_one(target_pdb, binder_len, iters=(20,20,5)):
    clear_mem(); m=mk_afdesign_model(protocol='binder')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=binder_len, hotspot=HOTSPOT)
    m.design_3stage(iters[0],iters[1],iters[2]); return m.get_seq()[0], _metrics(m)
def predict_seq(target_pdb, seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=len(seq), hotspot=HOTSPOT)
    m.set_seq(seq=seq); m.predict(verbose=False); return _metrics(m)
seq,mut_m=design_one(MUT_PDB, 70); wt_m=predict_seq(WT_PDB, seq)
print('SMOKE seq:', seq); print('MUT:', mut_m, '| WT:', wt_m)
print('[CELL 4]', '✓ SMOKE PASSED — real metrics; launch Cell 5' if mut_m.get('i_pae') is not None else '✗ fix before Cell 5')

In [ ]:
#@title Cell 5 — THE 4-HOUR BATCH (real designs, time-boxed + resumable). Run only if Cell 4 passed.
max_hours=4.0 #@param {type:'number'}
binder_len=70 #@param {type:'integer'}
import os, time, json, glob
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True)
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/metrics.json'))
print(f'[CELL 5] resuming from {i} done; batch until', time.strftime('%H:%M',time.localtime(t_end)))
PAE_BIND=10.0; PLDDT_MIN=80.0
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/metrics.json'): i+=1; continue
    os.makedirs(dd, exist_ok=True)
    try:
        seq,mut_m=design_one(MUT_PDB, binder_len)
        rec={'design_id':did,'target':'IDH1_R132H_A0101','sequence':seq,'mut':{'pae_interaction':mut_m['i_pae'],'binder_plddt':mut_m['plddt']}}
        if mut_m['i_pae']<=PAE_BIND and mut_m['plddt']>=PLDDT_MIN:   # only spend a WT fold on credible binders
            wt_m=predict_seq(WT_PDB, seq); rec['wt']={'pae_interaction':wt_m['i_pae']}
        json.dump(rec, open(f'{dd}/metrics.json','w'))
        print(f'  {did}: mut_ipae={mut_m["i_pae"]:.1f} plddt={mut_m["plddt"]:.0f}'+(f' wt_ipae={rec.get("wt",{}).get("pae_interaction",-1):.1f}' if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/metrics.json','w'))
    i+=1
print(f'[CELL 5] done — {i} designs attempted. Run Cell 6 to rank.')

In [ ]:
#@title Cell 6 — RANK + bundle (partial or complete)
import os, sys, json, glob, zipfile, shutil
WORK=os.environ['RUNG26_WORK']; DES=f'{WORK}/designs'
from scripts.runlog import run_logged, finalize
dst='runs/rung26_binder_design'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m,d)
run_logged([sys.executable,'-u','scripts/52_binder_design.py','rank', dst], RUNLOG)
if os.path.exists(f'{dst}/rung26_binder_design.json'):
    print(json.dumps(json.load(open(f'{dst}/rung26_binder_design.json'))['HEADLINE']))
finalize(RUNLOG, download=False); b='/content/rung26_binder_design.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True)+[str(RUNLOG)]:
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')